# MRIscanner: train on genuinely external data (BraTS + IXI)

The Kaggle "Brain Tumor MRI Dataset" this project trains on is, per its own
documented provenance, a combination of just 3 source collections from a narrow
set of scanners -- models trained on it generalize poorly to real-world scans
from elsewhere (works on Kaggle-style test scans, struggles on real online
photos). This notebook adds genuinely independent data to the 2 classes where a
free, no-registration source actually exists:

- **glioma**: [BraTS](https://www.med.upenn.edu/cbica/brats2020/data.html) -- 19 real
  institutions, T1-contrast-enhanced scans + tumor segmentation masks.
- **notumor**: [IXI](https://brain-development.org/ixi-dataset/) -- 3 real hospitals
  (Hammersmith/Guys/Institute of Psychiatry), healthy control scans.

meningioma/pituitary are **not** extended here: the only independent alternative
found (TCIA's Meningioma-SEG-CLASS) requires *you* to personally sign a
restricted-license data-use agreement with TCIA -- a real per-person legal step
that can't be done on your behalf from a notebook.

### One-time setup before you hit Run All
1. Runtime -> Change runtime type -> **T4 GPU** (or better, if you have Colab Pro --
   this run is much bigger than previous ones).
2. Get your `kaggle.json` (kaggle.com -> Settings -> API -> Create New Token --
   the file download, not the copyable token string) and drag it into `/content/`
   via the folder icon in the left sidebar.
3. **Runtime -> Run all.** This is a genuinely long job: downloading BraTS (~7GB) +
   IXI (~4GB) + the original Kaggle set, extracting thousands of slices, training
   all 3 architectures on a meaningfully larger dataset, and benchmarking against
   both the original held-out test set AND a fresh external-only held-out set.
   Expect multiple hours end to end -- that's the point, not a bug.


In [ ]:
!nvidia-smi


In [ ]:
import os, subprocess

REPO_DIR = "/content/MRIscanner"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "https://github.com/christiandrep7/MRIscanner.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
assert os.path.isdir("src"), f"expected src/ under {os.getcwd()} -- clone did not land where expected"
print("cwd:", os.getcwd())


### Install dependencies
Colab already ships a CUDA-enabled torch/torchvision -- skip the repo's CPU-era pins and install everything else (now includes nibabel, for reading the NIfTI-format BraTS/IXI volumes).

In [ ]:
!grep -v -E "^(torch|torchvision)==" requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt


### Kaggle credentials
Uses the `kaggle.json` you dropped into `/content/` -- no interactive prompt, so it doesn't block Run All.

In [ ]:
import os, shutil
from pathlib import Path

os.environ.pop("KAGGLE_API_TOKEN", None)
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

json_candidates = ["/content/kaggle.json", "kaggle.json"]
src = next((c for c in json_candidates if os.path.exists(c)), None)
if src is None:
    raise FileNotFoundError(
        "No kaggle.json found. Click the folder icon in the left sidebar, drag your "
        "kaggle.json into /content/, then Runtime > Run all again."
    )
shutil.copy(src, kaggle_dir / "kaggle.json")
os.chmod(kaggle_dir / "kaggle.json", 0o600)
print("Kaggle credentials installed from kaggle.json.")


### Download the original Kaggle dataset (data/Training, data/Testing)

In [ ]:
!python download_mri_dataset.py


### Download BraTS (glioma) and IXI (notumor) raw volumes
Only the files actually needed (BraTS: t1ce + seg; IXI: T1) -- but the Kaggle API downloads full datasets in one call, so this pulls everything and deletes the unused modalities (+ the leftover zip) right after extraction, before doing anything else. Real bug hit running this on Colab without this step: BraTS alone extracts to 25GB+ if all 5 modalities are kept, which exhausted the disk (`OSError: [Errno 28] No space left on device`) partway through the IXI download.

In [ ]:
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi
from run_external_data_pipeline import _reclaim_disk_after_brats_download, _reclaim_disk_after_ixi_download

api = KaggleApi()
api.authenticate()

brats_dir = Path("/content/external_raw/brats")
ixi_dir = Path("/content/external_raw/ixi")
brats_dir.mkdir(parents=True, exist_ok=True)
ixi_dir.mkdir(parents=True, exist_ok=True)

print("Downloading BraTS2020 (this is the big one, ~7GB compressed)...")
api.dataset_download_files(
    "awsaf49/brats20-dataset-training-validation",
    path=str(brats_dir), unzip=True, quiet=False,
)
_reclaim_disk_after_brats_download(brats_dir)

print("Downloading IXI (~4GB)...")
api.dataset_download_files(
    "wailrami/ixi-healthy-brain-mri-t1-t2",
    path=str(ixi_dir), unzip=True, quiet=False,
)
_reclaim_disk_after_ixi_download(ixi_dir)


### Extract 2D slices
BraTS: slices where the tumor segmentation mask shows real tumor, spread across each patient's tumor extent. IXI: evenly spaced slices from the central 60% of each healthy volume. Both reoriented to canonical (RAS+) before slicing -- verified during development that skipping this step can silently produce a sagittal slice instead of axial.

In [ ]:
from pathlib import Path
from src.external_data import extract_brats_glioma_slices, extract_ixi_notumor_slices

# extract_*_slices() already searches recursively for patient folders / .nii
# files, so the exact nesting Kaggle unzips into doesn't matter here.
n_glioma = extract_brats_glioma_slices(
    Path("/content/external_raw/brats"), Path("/content/external_raw_slices/glioma"), slices_per_patient=6
)
print(f"Extracted {n_glioma} new glioma slices from BraTS")

n_notumor = extract_ixi_notumor_slices(
    Path("/content/external_raw/ixi"), Path("/content/external_raw_slices/notumor"), slices_per_subject=4
)
print(f"Extracted {n_notumor} new notumor slices from IXI")

assert n_glioma > 100, (
    f"Only {n_glioma} glioma slices extracted -- something's off with the BraTS "
    "download/layout (check the unzip above actually produced BraTS20_Training_* folders)."
)
assert n_notumor > 100, (
    f"Only {n_notumor} notumor slices extracted -- something's off with the IXI "
    "download/layout (check the unzip above actually produced .nii files)."
)


In [ ]:
# Free up disk before training -- the raw BraTS/IXI downloads (t1/t2/flair
# modalities we never used, plus the now-extracted source volumes) aren't
# needed anymore and BraTS alone is several GB.
import shutil as _shutil
_shutil.rmtree("/content/external_raw", ignore_errors=True)
print("Freed raw download space.")


### Dedupe against the existing dataset
BraTS/IXI are independent sources, so real overlap is very unlikely -- but this reuses the project's own MD5 + perceptual-hash dedup (built for exactly this kind of check) as cheap insurance against test-set leakage before anything gets merged into data/Training.

In [ ]:
from src.secret_dedupe_eval import build_reference_index, dedupe_secret

reference_md5, reference_phash = build_reference_index([Path("data/Training"), Path("data/Testing")])
stats = dedupe_secret(
    secret_root=Path("/content/external_raw_slices"),
    out_root=Path("/content/external_deduped"),
    md5_set=reference_md5,
    ph_list=reference_phash,
    max_hamming=5,
)
print(stats)


### Merge: most of the new data into Training, a held-out slice into a fresh ExternalTesting set
`data/ExternalTesting` is never trained on -- it's the real test of whether this actually fixed cross-distribution generalization, since `data/Testing` is still Kaggle-style (same homogeneity, just a different split).

In [ ]:
import random

random.seed(42)
EXTERNAL_TEST_FRACTION = 0.2

for class_name in ["glioma", "notumor"]:
    src_dir = Path("/content/external_deduped") / class_name
    if not src_dir.exists():
        print(f"No deduped images for {class_name}, skipping")
        continue
    images = sorted(src_dir.glob("*.png"))
    random.shuffle(images)
    n_test = int(len(images) * EXTERNAL_TEST_FRACTION)
    test_images, train_images = images[:n_test], images[n_test:]

    train_dest = Path("data/Training") / class_name
    test_dest = Path("data/ExternalTesting") / class_name
    train_dest.mkdir(parents=True, exist_ok=True)
    test_dest.mkdir(parents=True, exist_ok=True)

    for p in train_images:
        p.rename(train_dest / p.name)
    for p in test_images:
        p.rename(test_dest / p.name)

    print(f"{class_name}: {len(train_images)} -> data/Training, {len(test_images)} -> data/ExternalTesting")


In [ ]:
# Sanity check: final class counts across both training and both test sets.
for split in ["Training", "Testing", "ExternalTesting"]:
    split_dir = Path("data") / split
    if not split_dir.exists():
        continue
    print(f"\n{split}:")
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        count = sum(1 for _ in class_dir.glob("*"))
        print(f"  {class_dir.name}: {count}")


### Train all 3 architectures on the enlarged dataset
Same recipe as before (augmentation, glioma loss-weighting, false-negative penalty) -- now on a meaningfully bigger, more diverse Training set. This is the genuinely multi-hour step.

In [ ]:
!python -m src.train_all


### Benchmark against the original data/Testing set (same as previous runs, for direct comparison)

In [ ]:
!python -m src.benchmark


### Benchmark against data/ExternalTesting -- the real test
This is BraTS/IXI-only data the models never trained on. If the external data actually helped cross-distribution generalization, glioma/notumor recall here should look meaningfully better than a model trained on Kaggle data alone would.

In [ ]:
from src.config import DataConfig
from src.benchmark import default_checkpoint_specs, benchmark_models, save_benchmark_report

external_cfg = DataConfig(test_dir_name="ExternalTesting")
specs = default_checkpoint_specs(Path("checkpoints"))
external_results = benchmark_models(specs, external_cfg, output_dir=Path("outputs/benchmark_external"))
save_benchmark_report(external_results, Path("outputs/benchmark_external"))
for r in external_results:
    if r.available:
        print(f"{r.architecture}: accuracy={r.accuracy:.4f} macro_f1={r.macro_f1:.4f}")
    else:
        print(f"{r.architecture}: unavailable ({r.error})")


### Glioma numbers side by side: original Testing vs the new external-only ExternalTesting

In [ ]:
import json

for label, bench_dir in [("data/Testing", "outputs/benchmark"), ("data/ExternalTesting", "outputs/benchmark_external")]:
    print(f"--- {label} ---")
    for arch in ["resnet50", "efficientnet_b0", "vgg16"]:
        report_path = Path(bench_dir) / arch / "classification_report.json"
        if not report_path.exists():
            continue
        report = json.load(open(report_path))
        g = report.get("glioma")
        if g:
            print(
                f"  {arch:16s} glioma recall={g['recall']:.4f}",
                f"precision={g['precision']:.4f}",
                f"overall_acc={report['accuracy']:.4f}",
            )


### Confusion matrices for the external-only test set

In [ ]:
from IPython.display import Image as IPImage, display

for arch in ["resnet50", "efficientnet_b0", "vgg16"]:
    path = Path("outputs/benchmark_external") / arch / "confusion_matrix.png"
    if path.exists():
        print(arch)
        display(IPImage(filename=str(path)))


### Download checkpoints + both benchmark reports back to your machine

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/mriscanner_results", "zip", ".", "checkpoints")
shutil.make_archive("/content/mriscanner_outputs", "zip", ".", "outputs")
files.download("/content/mriscanner_results.zip")
files.download("/content/mriscanner_outputs.zip")
